# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates how to load and explore the [FAIR^2](https://doi.org/10.71728/senscience.qs2f-h81p) dataset using the `mlcroissant` library, referencing all entities by their Croissant `@id` fields.

### Dataset Source

The dataset source is provided via a Croissant schema URL:  
`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
import warnings
warnings.filterwarnings("ignore")

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their Croissant `@id`s.

Each record set, field, and column is referenced by its `@id` for unambiguous selection and extraction.

In [ ]:
# List all available record sets in the dataset by their @id
record_sets = dataset.record_sets
print("Found record sets:")
for rs in record_sets:
    print(f"- {rs['@id']}: {rs['name'] if 'name' in rs else '(no name)'}")

# Let's select the first available record set for demonstration
if len(record_sets) == 0:
    raise ValueError("No record sets found in the Croissant schema.")
record_set_id = record_sets[0]['@id']

print(f"\nFields and columns for record set @id: {record_set_id}")
fields = record_sets[0]['field']
if isinstance(fields, dict):
    fields = [fields]
for field in fields:
    field_id = field['@id']
    col_ids = []
    if 'column' in field:
        if isinstance(field['column'], list):
            col_ids = [col['@id'] for col in field['column']]
        else:
            col_ids = [field['column']['@id']]
    print(f"  - Field @id: {field_id} | Columns: {col_ids}")

## 3. Data Extraction
Load data from the record set into a DataFrame using its Croissant `@id`. All fields are handled exclusively by their `@id` values.


In [ ]:
# Extract records for all record sets into dataframes by their @id
all_record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

for rs_id in all_record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    if records:
        dataframes[rs_id] = pd.DataFrame(records)

# Print the columns of the first record set
print(f"Columns for record set @id {record_set_id}:")
if record_set_id in dataframes:
    print(dataframes[record_set_id].columns.tolist())
    dataframes[record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps: filtering, normalization, grouping, etc. All fields are referenced using their Croissant `@id`s.

In [ ]:
# Choose a numeric field by its @id for demonstration
df = dataframes[record_set_id]

# Attempt to infer a numeric field @id (column) from schema or data
# For robust notebooks, you'd document the field selection using prior cell outputs
numeric_field_id = None
for col in df.columns:
    if pd.api.types.is_numeric_dtype(df[col]):
        numeric_field_id = col
        break
if not numeric_field_id:
    print('No numeric field found in this record set; adjust field selection as appropriate.')
else:
    threshold = df[numeric_field_id].mean() if pd.notnull(df[numeric_field_id].mean()) else 10
    
    # Filter records where the numeric field exceeds the threshold
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize the numeric column (z-score)
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Optionally group by another field (try to select a string/categorical field)
    group_field_id = None
    for col in df.columns:
        if col != numeric_field_id and pd.api.types.is_object_dtype(df[col]):
            group_field_id = col
            break
    if group_field_id:
        grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame(name=f"mean_{numeric_field_id}")
        print(f"\nGrouped mean of {numeric_field_id} by {group_field_id}:")
        print(grouped.head())

## 5. Visualization
Visualize a numeric field distribution or the relationship between two fields using the columns' `@id`s.

If visualization libraries (matplotlib, seaborn) are not installed, install them as needed.

In [ ]:
# Visualization example: histogram for selected numeric field by @id
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), bins=16, kde=True, color="tab:blue")
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()
    
    # If a grouping field is available, use boxplot
    if group_field_id:
        plt.figure(figsize=(10, 4))
        sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=30, ha='right')
        plt.show()

## 6. Conclusion

- We demonstrated how to use `mlcroissant` to load, explore, and process clinical cancer data fully referenced by Croissant `@id` fields.
- Key steps included getting record sets/fields `@id` values, extracting data, filtering/normalizing by field `@id`, and visualizing numeric features.
- For deeper analysis, refer to the dataset's full Croissant schema for available fields and @id mappings.

_End of notebook._